# Flash Flash Revolution Silver Layer HorizontalDensity Calculation

This notebook calculates **HorizontalDensity**, a metric measuring the temporal clustering of notes across all orientations. Unlike VerticalDensity (which measures same-orientation note intensity), HorizontalDensity uses a convolution-like weighted approach to quantify how densely notes are packed in time, regardless of orientation.

**Key Concept:**
For each note, we calculate a weighted sum density by examining all surrounding notes (including the note itself) within a time window and applying weights based on **signed temporal proximity**. The weights form a kernel function that emphasizes notes closer in time.

**Algorithm:**
1. For each note (target), find all other notes within ±117ms **AND include the note itself**
2. Calculate **signed** time difference: `target.timestamp - nearby.timestamp`
   - Self (time_diff = 0) falls in `[-17ms, +17ms)` interval → weight 1.0
   - Negative = nearby note comes BEFORE target
   - Positive = nearby note comes AFTER target
3. Assign weight based on partition interval:
   - `[-117ms, -83ms)`: weight 0.1 (notes far before)
   - `[-83ms, -50ms)`: weight 0.5 (notes medium before)
   - `[-50ms, -17ms)`: weight 1.0 (notes close before)
   - `[-17ms, +17ms)`: weight 1.0 (notes very close + self)
   - `[+17ms, +50ms)`: weight 1.0 (notes close after)
   - `[+50ms, +83ms)`: weight 0.5 (notes medium after)
   - `[+83ms, +117ms)`: weight 0.5 (notes far after)
   - Outside window: weight 0.0
4. **Sum** all weights (including self) → this is the `horizontal_density` score

**Density Interpretation:**
- **Minimum density = 1.0**: Isolated notes with no neighbors (only the note itself)
- **Higher density**: Notes with nearby notes have sum > 1.0
- The metric captures both the number and temporal proximity of surrounding notes

**Key Features:**
- **Note Decomposition**: Multi-orientation notes decomposed into single orientations
- **Self-Inclusion**: Each note includes itself with weight 1.0 (base density)
- **Incremental Processing**: Uses `swf_version` tracking to detect new/updated songs
- **Note-Level Data**: Stores HorizontalDensity for each note
- **Configurable Parameters**: Partition boundaries and weights can be adjusted
- **Signed Time Differences**: Preserves temporal directionality

**Source Table:**
- `acubed.ffr.silver__notes-adjusted`: Note-level data with timestamps and orientations

**Target Table:**
- `acubed.ffr.silver__horizontal-density`: Note-level HorizontalDensity data

**Schema:**
- `song_id` (bigint): Unique song identifier
- `note_id` (int): Note sequence number
- `timestamp` (double): Note timestamp in seconds
- `orientation` (string): Single orientation after decomposition
- `horizontal_density` (double): Sum of weights (minimum 1.0)
- `nearby_notes_count` (int): Number of OTHER notes within time window (excludes self)
- `swf_version` (long): Version tracking for incremental updates

In [0]:
# Judge window configuration (in seconds)
# These parameters define partition boundaries and weights for the convolution kernel

# Partition boundaries (in seconds) - defines the bin edges
# Creates intervals: [partition[i], partition[i+1])
PARTITION_BOUNDARIES = [-0.117, -0.083, -0.05, -0.017, 0.017, 0.05, 0.083, 0.117]

# Weights for each interval between boundaries
# weights[i] applies to interval [partition[i], partition[i+1])
WEIGHTS = [0.1, 0.5, 1.0, 1.0, 1.0, 0.5, 0.5]

# Convert to milliseconds for display
PARTITION_MS = [b * 1000 for b in PARTITION_BOUNDARIES]
MAX_WINDOW_MS = max(abs(b) for b in PARTITION_MS)

print("✓ Judge window configuration loaded")
print("\nPartition intervals and weights:")
for i in range(len(WEIGHTS)):
    start_ms = PARTITION_MS[i]
    end_ms = PARTITION_MS[i+1]
    weight = WEIGHTS[i]
    print(f"  [{start_ms:6.1f}ms, {end_ms:6.1f}ms): weight {weight:.1f}")
print(f"\nMaximum window: ±{MAX_WINDOW_MS:.1f}ms")
print(f"Notes outside [±{MAX_WINDOW_MS:.1f}ms] receive weight 0.0")

In [0]:
# Configuration: Automatic Processing Mode
# Automatically determines whether to do full refresh or incremental processing
# - Full refresh (process_all=True): When table doesn't exist (first run)
# - Incremental (process_all=False): When table exists (subsequent runs)

silver_table = "acubed.ffr.`silver__horizontal-density`"
process_all = not spark.catalog.tableExists(silver_table)

if process_all:
    print(f"ℹ Auto-detected: Full refresh mode (table does not exist)")
else:
    print(f"ℹ Auto-detected: Incremental mode (table exists)")

print(f"✓ Configuration loaded: process_all = {process_all}")

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import (
    col, explode, array, when, lit, count, mean, expr, abs as spark_abs, sum as spark_sum
)
from delta.tables import DeltaTable

print("✓ Imports loaded successfully")

In [0]:
# Identify songs that need to be processed (new or updated)
# Uses swf_version from bronze__songlist to detect changes

silver_table = "acubed.ffr.`silver__horizontal-density`"

if spark.catalog.tableExists(silver_table) and not process_all:
    # Silver table exists - check for new songs and updated songs
    bronze_songlist = spark.table("acubed.ffr.bronze__songlist").select(
        col("id").alias("song_id"),
        col("swf_version").alias("bronze_swf_version")
    )
    
    silver_horizontal_density = spark.table(silver_table).select(
        col("song_id").alias("silver_song_id"),
        col("swf_version").alias("silver_swf_version")
    ).distinct()
    
    # Left join to find new songs and updated songs
    changed_songs = bronze_songlist.join(
        silver_horizontal_density,
        bronze_songlist.song_id == silver_horizontal_density.silver_song_id,
        "left"
    ).filter(
        # New songs (not in silver) OR updated songs (different swf_version)
        col("silver_song_id").isNull() |
        (col("bronze_swf_version") != col("silver_swf_version"))
    )
    
    changed_song_ids = changed_songs.select(col("song_id")).distinct()
    num_changed = changed_song_ids.count()
    
    if num_changed == 0:
        print("ℹ No changed songs detected")
        changed_song_ids = None
    else:
        print(f"✓ Found {num_changed} songs to process (new or updated)")
else:
    if process_all:
        print("ℹ Force full refresh mode enabled (process_all=True)")
    else:
        print("ℹ First run - no existing silver table found")
    print("✓ Will process all songs")
    changed_song_ids = None  # Set to None for full refresh

In [0]:
# Determine processing mode based on change detection  
if not process_all and changed_song_ids is None:
    print("ℹ No songs to process - skipping HorizontalDensity calculation")
    print("✓ Notebook will complete successfully (idempotent run)")
    # Set empty DataFrame to allow downstream cells to run
    df_density_with_version = spark.createDataFrame([], schema="song_id long, note_id int, timestamp double, orientation string, horizontal_density double, nearby_notes_count int, swf_version long")
else:
    # Load silver notes adjusted table
    df_notes = spark.table("acubed.ffr.`silver__notes-adjusted`")
    
    # Filter to changed songs if doing incremental processing
    if not process_all and changed_song_ids is not None:
        df_notes = df_notes.join(changed_song_ids, "song_id", "inner")
        print(f"ℹ Incremental mode: Processing {df_notes.select('song_id').distinct().count()} changed songs")
    else:
        print(f"ℹ Full refresh mode: Processing all {df_notes.select('song_id').distinct().count()} songs")
    
    print("\nStep 1: Decomposing multi-orientation notes...")
    print("=" * 60)
    
    # Decompose multi-orientation notes into separate rows
    df_decomposed = df_notes.select(
        "song_id",
        "note_id",
        "timestamp",
        "orientation"
    ).withColumn(
        "single_orientations",
        array(
            when(col("orientation").substr(1, 1) == "1", lit("1000")).otherwise(lit(None)),
            when(col("orientation").substr(2, 1) == "1", lit("0100")).otherwise(lit(None)),
            when(col("orientation").substr(3, 1) == "1", lit("0010")).otherwise(lit(None)),
            when(col("orientation").substr(4, 1) == "1", lit("0001")).otherwise(lit(None))
        )
    ).select(
        "song_id",
        "note_id",
        "timestamp",
        explode("single_orientations").alias("orientation")
    ).filter(
        col("orientation").isNotNull()
    )
    
    print(f"✓ Decomposed notes into single orientations")
    
    print("\nStep 2: Self-join to find nearby notes within time window...")
    print("=" * 60)
    
    # Self-join to find all notes within time window for each note
    MAX_WINDOW = max(abs(b) for b in PARTITION_BOUNDARIES)
    
    df_nearby = df_decomposed.alias("target").join(
        df_decomposed.alias("nearby"),
        (col("target.song_id") == col("nearby.song_id")) &
        (spark_abs(col("target.timestamp") - col("nearby.timestamp")) <= MAX_WINDOW),
        "inner"
    ).select(
        col("target.song_id"),
        col("target.note_id"),
        col("target.timestamp"),
        col("target.orientation"),
        col("nearby.timestamp").alias("nearby_timestamp")
    )
    
    print(f"✓ Found nearby notes within ±{MAX_WINDOW}ms window")
    
    print("\nStep 3: Calculating weighted density...")
    print("=" * 60)
    
    # Calculate signed time difference
    df_with_time_diff = df_nearby.withColumn(
        "time_diff",
        col("timestamp") - col("nearby_timestamp")
    )
    
    # Assign weights based on partition intervals
    weight_expr = lit(0.0)  # Default weight for outside window
    for i in range(len(WEIGHTS)):
        lower_bound = PARTITION_BOUNDARIES[i]
        upper_bound = PARTITION_BOUNDARIES[i+1]
        weight = WEIGHTS[i]
        
        weight_expr = when(
            (col("time_diff") >= lower_bound) & (col("time_diff") < upper_bound),
            lit(weight)
        ).otherwise(weight_expr)
    
    df_weighted = df_with_time_diff.withColumn("weight", weight_expr)
    
    # Aggregate weights to get horizontal density
    df_density = df_weighted.groupBy(
        "song_id",
        "note_id",
        "timestamp",
        "orientation"
    ).agg(
        spark_sum("weight").alias("horizontal_density"),
        (count("*") - 1).alias("nearby_notes_count")  # Exclude self from count
    )
    
    print(f"✓ Calculated horizontal_density for {df_density.count()} notes")
    
    print("\nStep 4: Adding swf_version for change tracking...")
    print("=" * 60)
    
    # Join with bronze_songlist to add swf_version
    bronze_songlist = spark.table("acubed.ffr.bronze__songlist").select(
        col("id").alias("song_id"),
        col("swf_version")
    )
    
    df_density_with_version = df_density.join(
        bronze_songlist,
        "song_id",
        "inner"
    ).select(
        "song_id",
        "note_id",
        "timestamp",
        "orientation",
        "horizontal_density",
        "nearby_notes_count",
        "swf_version"
    )
    
    print(f"✓ Added swf_version to {df_density_with_version.count()} notes")
    print("\nSample output:")
    display(df_density_with_version.orderBy("song_id", "note_id").limit(10))

In [0]:
# Save to Delta table using DELETE + INSERT pattern for incremental updates
table_name = "acubed.ffr.`silver__horizontal-density`"

# Initialize variable for tracking deletions
song_ids_to_delete = []


# Capture counts BEFORE operations
rows_to_insert = df_density_with_version.count()

if rows_to_insert == 0:
    print("ℹ No rows to insert - skipping table update")
    print("✓ Table remains unchanged (idempotent run)")
else:
    if spark.catalog.tableExists(table_name):
        # Table exists - use DELETE + INSERT for incremental updates
        delta_table = DeltaTable.forName(spark, table_name)
        
        # Get list of song_ids being updated
        updated_song_ids = df_density_with_version.select("song_id").distinct()
        
        # Delete existing rows for these songs
        song_ids_to_delete = [row.song_id for row in updated_song_ids.collect()]
        
        if len(song_ids_to_delete) > 0:
            # Build condition for DELETE
            delete_condition = col("song_id").isin(song_ids_to_delete)
            delta_table.delete(delete_condition)
            print(f"✓ Deleted existing rows for {len(song_ids_to_delete)} songs")
        
        # Insert new rows
        df_density_with_version.write.format("delta").mode("append").saveAsTable(table_name)
        print(f"✓ Inserted {rows_to_insert} new rows")
    else:
        # First run - create table
        df_density_with_version.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(table_name)
        print(f"✓ Created table {table_name} with {rows_to_insert} rows")
    
    print(f"\n✓ Updated {len(song_ids_to_delete) if spark.catalog.tableExists(table_name) and len(song_ids_to_delete) > 0 else 'all'} songs in {table_name}")

print("\n" + "=" * 60)
print("HorizontalDensity calculation complete!")
print("=" * 60)

In [0]:
import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql.functions import col, mean

# Load the full horizontal-density table (note-level data)
df_horizontal_density = spark.table("acubed.ffr.`silver__horizontal-density`")

# Aggregate to song level for visualization
df_song_avg = df_horizontal_density.groupBy("song_id").agg(
    mean("horizontal_density").alias("avg_density")
)

# Load song metadata
df_songlist = spark.table("acubed.ffr.bronze__songlist")

# Join with song metadata to get difficulty
df_density_difficulty = df_song_avg.join(
    df_songlist.select("id", "difficulty", "name"),
    df_song_avg.song_id == df_songlist.id,
    "inner"
).select(
    col("song_id"),
    col("name"),
    col("avg_density"),
    col("difficulty")
)

print("=" * 70)
print("HORIZONTALDENSITY vs DIFFICULTY VISUALIZATION")
print("=" * 70)

# Check for nulls and basic stats
total_songs = df_density_difficulty.count()
songs_with_difficulty = df_density_difficulty.filter(col("difficulty").isNotNull()).count()

print(f"\nTotal songs analyzed: {total_songs:,}")
print(f"Songs with difficulty data: {songs_with_difficulty:,}")

if songs_with_difficulty > 0:
    # Convert to pandas for plotting
    df_plot = df_density_difficulty.filter(col("difficulty").isNotNull()).toPandas()
    
    print(f"\nDifficulty range: {df_plot['difficulty'].min()} to {df_plot['difficulty'].max()}")
    print(f"Avg HorizontalDensity range: {df_plot['avg_density'].min():.2f} to {df_plot['avg_density'].max():.2f}")
    
    # Create scatterplot
    fig, ax = plt.subplots(figsize=(14, 10))
    
    # Color by difficulty for better visualization
    scatter = ax.scatter(
        df_plot['avg_density'], 
        df_plot['difficulty'],
        alpha=0.6,
        s=50,
        c=df_plot['difficulty'],
        cmap='RdYlGn_r',  # Red (hard) to Green (easy)
        edgecolors='black',
        linewidth=0.5
    )
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax)
    cbar.set_label('Difficulty', fontsize=12)
    
    # Set axis limits (adjusted for sum-based density)
    ax.set_xlim(0, 5)
    ax.set_ylim(0, 150)
    
    # Labels and title
    ax.set_xlabel('Average HorizontalDensity (sum of weighted note density)', fontsize=12, fontweight='bold')
    ax.set_ylabel('Difficulty', fontsize=12, fontweight='bold')
    ax.set_title('Song Difficulty vs Average HorizontalDensity\n(Convolution-based Weighted Sum, includes self with base 1.0)', 
                 fontsize=14, fontweight='bold', pad=20)
    
    # Add grid
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Calculate and display correlation for visible range
    df_visible = df_plot[(df_plot['avg_density'] >= 0) & (df_plot['avg_density'] <= 5) & (df_plot['difficulty'] <= 150)]
    correlation = df_visible['avg_density'].corr(df_visible['difficulty'])
    ax.text(0.02, 0.98, f'Correlation: {correlation:.3f}\n(range: 0-5, 0-150)\nMin density: 1.0', 
            transform=ax.transAxes, fontsize=12, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    plt.tight_layout()
    plt.show()
    
    print("\n✓ Visualization complete!")
    print(f"   Correlation: {correlation:.3f}")
    print(f"   Songs in visible range: {len(df_visible):,} / {len(df_plot):,} ({len(df_visible)/len(df_plot)*100:.1f}%)")
else:
    print("\n⚠️ No songs with difficulty data found!")

In [0]:
from pyspark.sql.functions import col, abs as spark_abs, when, lit
import pandas as pd

print("=" * 70)
print("EXAMPLE: HORIZONTAL DENSITY CALCULATION FOR A SINGLE NOTE")
print("=" * 70)

# Pick a specific note to demonstrate the calculation
# Let's choose song_id=1, note_id=200, orientation="1000" (left arrow)
example_song_id = 1
example_note_id = 200
example_orientation = "1000"

print(f"\nTarget Note: song_id={example_song_id}, note_id={example_note_id}, orientation={example_orientation}")

# Get all decomposed notes for this song
df_song_notes = spark.table("acubed.ffr.`silver__notes-adjusted`").filter(col("song_id") == example_song_id)

# Decompose multi-orientation notes
df_song_decomposed = df_song_notes.select(
    "song_id",
    "note_id",
    "timestamp",
    "orientation"
).withColumn(
    "single_orientations",
    array(
        when(col("orientation").substr(1, 1) == "1", lit("1000")),
        when(col("orientation").substr(2, 1) == "1", lit("0100")),
        when(col("orientation").substr(3, 1) == "1", lit("0010")),
        when(col("orientation").substr(4, 1) == "1", lit("0001"))
    )
).withColumn(
    "single_orientations",
    expr("filter(single_orientations, x -> x is not null)")
).withColumn(
    "decomposed_orientation",
    explode(col("single_orientations"))
).select(
    "song_id",
    "note_id",
    "timestamp",
    col("decomposed_orientation").alias("orientation")
)

# Get the target note details
target_note = df_song_decomposed.filter(
    (col("note_id") == example_note_id) & 
    (col("orientation") == example_orientation)
).first()

if target_note is None:
    print("\n⚠️ Target note not found. Let's pick a different note...")
    # Get first note from the table
    target_note = df_song_decomposed.filter(col("orientation") == "1000").first()
    example_note_id = target_note.note_id
    print(f"Using: song_id={example_song_id}, note_id={example_note_id}, orientation={example_orientation}")

target_timestamp = target_note.timestamp
print(f"Target timestamp: {target_timestamp:.3f} seconds")

# Find all nearby notes within ±117ms window (INCLUDING THE TARGET NOTE ITSELF)
MAX_WINDOW_SEC = 0.117

df_nearby = df_song_decomposed.filter(
    spark_abs(col("timestamp") - target_timestamp) <= MAX_WINDOW_SEC
).withColumn(
    "time_diff_sec",
    lit(target_timestamp) - col("timestamp")  # SIGNED difference
).withColumn(
    "time_diff_ms",
    col("time_diff_sec") * 1000  # Convert to milliseconds for display
).withColumn(
    "is_self",
    when((col("note_id") == example_note_id) & (col("orientation") == example_orientation), lit("[SELF]")).otherwise(lit(""))
)

# Apply weight calculation based on partition boundaries
PARTITION_BOUNDARIES = [-0.117, -0.083, -0.05, -0.017, 0.017, 0.05, 0.083, 0.117]
WEIGHTS = [0.1, 0.5, 1.0, 1.0, 1.0, 0.5, 0.5]

weight_expr = when(
    (col("time_diff_sec") >= PARTITION_BOUNDARIES[0]) & 
    (col("time_diff_sec") < PARTITION_BOUNDARIES[1]),
    lit(WEIGHTS[0])
)
for i in range(1, len(WEIGHTS)):
    weight_expr = weight_expr.when(
        (col("time_diff_sec") >= PARTITION_BOUNDARIES[i]) & 
        (col("time_diff_sec") < PARTITION_BOUNDARIES[i+1]),
        lit(WEIGHTS[i])
    )
weight_expr = weight_expr.otherwise(lit(0.0))

df_nearby_weighted = df_nearby.withColumn("weight", weight_expr).orderBy("timestamp")

nearby_count = df_nearby_weighted.count()
print(f"\nAll notes within ±117ms window (including self): {nearby_count}")

if nearby_count > 0:
    # Convert to pandas for display
    df_display = df_nearby_weighted.select(
        "note_id",
        "orientation",
        "timestamp",
        "time_diff_ms",
        "weight",
        "is_self"
    ).toPandas()
    
    print("\n" + "=" * 70)
    print("ALL NOTES AND THEIR WEIGHTS (INCLUDING SELF)")
    print("=" * 70)
    print("\ntime_diff_ms interpretation:")
    print("  0.0 = target note itself (always weight 1.0)")
    print("  Negative = nearby note is BEFORE target note")
    print("  Positive = nearby note is AFTER target note")
    print("\n")
    
    # Format the display
    df_display['timestamp'] = df_display['timestamp'].round(3)
    df_display['time_diff_ms'] = df_display['time_diff_ms'].round(1)
    
    print(df_display.to_string(index=False))
    
    # Calculate horizontal_density (sum of weights including self)
    sum_weights = df_display['weight'].sum()
    num_neighbors = len(df_display) - 1  # Exclude self from neighbor count
    
    print("\n" + "=" * 70)
    print("CALCULATION")
    print("=" * 70)
    print(f"\nWeights: {df_display['weight'].tolist()}")
    print(f"Sum of weights (including self): {sum_weights:.3f}")
    print(f"Number of OTHER notes (neighbors): {num_neighbors}")
    print(f"\nhorizontal_density = sum of all weights (including self)")
    print(f"horizontal_density = {sum_weights:.4f}")
    print(f"\nNote: Minimum density is 1.0 (isolated note with only itself)")
    
    # Verify against actual table value
    actual_density = spark.table("acubed.ffr.`silver__horizontal-density`").filter(
        (col("song_id") == example_song_id) &
        (col("note_id") == example_note_id) &
        (col("orientation") == example_orientation)
    ).select("horizontal_density", "nearby_notes_count").first()
    
    if actual_density:
        print(f"\n" + "=" * 70)
        print("VERIFICATION")
        print("=" * 70)
        print(f"Calculated horizontal_density: {sum_weights:.4f}")
        print(f"Stored horizontal_density:     {actual_density.horizontal_density:.4f}")
        print(f"Stored nearby_notes_count:     {actual_density.nearby_notes_count} (excludes self)")
        
        if abs(sum_weights - actual_density.horizontal_density) < 0.001:
            print("\n✓ Calculation matches stored value!")
        else:
            print("\n⚠️ Calculation does not match stored value")
else:
    print("\n⚠️ No notes found in the time window.")

print("\n" + "=" * 70)
print("WEIGHT PARTITION REFERENCE")
print("=" * 70)
print("\nTime difference ranges and their weights:")
for i in range(len(WEIGHTS)):
    start_ms = PARTITION_BOUNDARIES[i] * 1000
    end_ms = PARTITION_BOUNDARIES[i+1] * 1000
    weight = WEIGHTS[i]
    interval_note = "(includes self at 0.0ms)" if start_ms < 0 and end_ms > 0 else ""
    print(f"  [{start_ms:6.1f}ms, {end_ms:6.1f}ms): weight {weight:.1f} {interval_note}")